In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
import geojson_validator
from shapely.ops import unary_union
import pandas as pd
import numpy as np
from shapely.geometry import Point, MultiPolygon
import geopandas as gpd
from geopandas import GeoDataFrame
from fuzzywuzzy import process

In [2]:
tehsil_gdf = gpd.read_file(r"D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\HP_tenders_dashboard\fixed_RHR_dupVertices_4326_reduced.geojson")
tehsil_ref = gpd.read_file(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\HP\flood-data-ecosystem-Himachal-Pradesh\Maps\HP_IDS-DRR_shapefiles\hp_tehsil_final.geojson')
risk_df = pd.read_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Deployment\IDS-DRR-Himachal-Pradesh-Risk-Score-Model\RiskScoreModel\data\risk_score_final.csv')

In [3]:
tehsil_merge = pd.merge(tehsil_gdf, tehsil_ref[['object_id','TEHSIL']],on='TEHSIL',how='left')
tehsil_gdf = tehsil_merge

In [15]:
tehsil_gdf

,District,STATE,TEHSIL,Shape_Leng,Shape_Area,geometry,object_id
0,BILASPUR,HIMACHAL PRADESH,BHARARI,50253.228763,9.000306e+07,"POLYGON ((76.72933 31.58233, 76.72061 31.58746...",02-030-02104
1,BILASPUR,HIMACHAL PRADESH,BILASPUR,93446.685818,1.703525e+08,"POLYGON ((76.87543 31.35065, 76.87222 31.35589...",02-030-02105
2,BILASPUR,HIMACHAL PRADESH,GHUMARWIN,105834.665205,1.906542e+08,"POLYGON ((76.78539 31.50015, 76.78270 31.50446...",02-030-02106
3,BILASPUR,HIMACHAL PRADESH,JHANDUTA,112831.475031,3.205376e+08,"POLYGON ((76.53093 31.46349, 76.52206 31.46352...",02-030-02107
4,BILASPUR,HIMACHAL PRADESH,NAMHOL,61752.616994,1.034979e+08,"POLYGON ((76.87507 31.33080, 76.86912 31.33225...",02-030-02108
...,...,...,...,...,...,...,...
107,KINNAUR,HIMACHAL PRADESH,HANGRANG,126162.028723,5.002174e+08,"POLYGON ((78.74985 31.90206, 78.75014 31.90448...",02-034-01703
108,KINNAUR,HIMACHAL PRADESH,MORANG,286156.570410,1.751222e+09,"POLYGON ((78.17226 31.86092, 78.16269 31.85914...",02-034-01704
109,KINNAUR,HIMACHAL PRADESH,POOH,212682.456325,1.217420e+09,"POLYGON ((78.45749 31.94192, 78.45636 31.93768...",02-034-01705
110,KINNAUR,HIMACHAL PRADESH,SANGLA,280578.805650,1.424896e+09,"POLYGON ((78.15690 31.51423, 78.14240 31.51981...",02-034-01706


In [4]:
# Function to fix invalid geometries
def fix_geometry(geom):
    if geom is None:
        return None
    if not geom.is_valid:
        # Attempt to fix using buffer(0)
        geom = geom.buffer(0)
    return geom if geom.is_valid else None

# Check and fix geometries
def clean_geometries(gdf):
    # Check for missing or invalid geometries
    gdf['geometry_fixed'] = gdf['geometry'].apply(fix_geometry)

    # Drop rows with irreparable (None) geometries
    gdf = gdf.dropna(subset=['geometry_fixed'])

    return gdf

def geometry_to_text(geom):
    if geom is None:
        return None
    
    # Handle Polygon and LineString geometries
    if geom.geom_type == 'Polygon':
        coords = list(geom.exterior.coords)
        return str([[lon, lat] for lon, lat in coords])
    
    elif geom.geom_type == 'LineString':
        coords = list(geom.coords)
        return str([[lon, lat] for lon, lat in coords])
    
    # Handle MultiPolygon and MultiLineString geometries
    elif geom.geom_type in ['MultiPolygon', 'MultiLineString']:
        all_coords = []
        for part in geom.geoms:  # Loop through each sub-geometry
            if part.geom_type == 'Polygon':
                coords = list(part.exterior.coords)
            else:
                coords = list(part.coords)
            all_coords.append([[lon, lat] for lon, lat in coords])
        return str(all_coords)
    
    # Catch other geometry types if necessary
    else:
        return None

def get_best_match(block_name, block_names):
    match, score = process.extractOne(block_name, block_names)
    return match if score > 80 else None  # Adjust threshold as needed

In [5]:
import json

# Flatten nested geometries if required
fixed_geo_flattened = {
    key: [json.dumps(item) if isinstance(item, dict) else item for item in value]
    for key, value in tehsil_gdf.items()
}
geo_fixed = pd.DataFrame.from_dict(fixed_geo_flattened)

In [6]:
# Simplify MultiPolygon by selecting the largest Polygon
def simplify_multipolygon(geometry):
    if isinstance(geometry, MultiPolygon):
        # Select the largest Polygon by area
        return max(geometry.geoms, key=lambda geom: geom.area)
    return geometry

# Apply the simplification
tehsil_gdf['geometry'] = tehsil_gdf['geometry'].apply(simplify_multipolygon)

geo_fixed = gpd.GeoDataFrame(tehsil_gdf, geometry='geometry')


In [7]:
tehsil_gdf

,District,STATE,TEHSIL,Shape_Leng,Shape_Area,geometry,object_id
0,BILASPUR,HIMACHAL PRADESH,BHARARI,50253.228763,9.000306e+07,"POLYGON ((76.72933 31.58233, 76.72061 31.58746...",02-030-02104
1,BILASPUR,HIMACHAL PRADESH,BILASPUR,93446.685818,1.703525e+08,"POLYGON ((76.87543 31.35065, 76.87222 31.35589...",02-030-02105
2,BILASPUR,HIMACHAL PRADESH,GHUMARWIN,105834.665205,1.906542e+08,"POLYGON ((76.78539 31.50015, 76.78270 31.50446...",02-030-02106
3,BILASPUR,HIMACHAL PRADESH,JHANDUTA,112831.475031,3.205376e+08,"POLYGON ((76.53093 31.46349, 76.52206 31.46352...",02-030-02107
4,BILASPUR,HIMACHAL PRADESH,NAMHOL,61752.616994,1.034979e+08,"POLYGON ((76.87507 31.33080, 76.86912 31.33225...",02-030-02108
...,...,...,...,...,...,...,...
107,KINNAUR,HIMACHAL PRADESH,HANGRANG,126162.028723,5.002174e+08,"POLYGON ((78.74985 31.90206, 78.75014 31.90448...",02-034-01703
108,KINNAUR,HIMACHAL PRADESH,MORANG,286156.570410,1.751222e+09,"POLYGON ((78.17226 31.86092, 78.16269 31.85914...",02-034-01704
109,KINNAUR,HIMACHAL PRADESH,POOH,212682.456325,1.217420e+09,"POLYGON ((78.45749 31.94192, 78.45636 31.93768...",02-034-01705
110,KINNAUR,HIMACHAL PRADESH,SANGLA,280578.805650,1.424896e+09,"POLYGON ((78.15690 31.51423, 78.14240 31.51981...",02-034-01706


In [8]:
merged_gdf = risk_df.merge(geo_fixed[['geometry','object_id']], left_on='object-id', right_on='object_id', how='left')
#merged_gdf = merged_gdf.drop(columns=['dtcode11','block_lgd','dtname','block_name'])
merged_gdf

,object-id,tehsil-area,district,timeperiod,total-tender-awarded-value,immediate-measures-tenders-awarded-value,others-tenders-awarded-value,objectid,max-rain,mean-rain,...,drainage-density,exposure,flood-hazard,government-response,vulnerability,topsis-score,risk-score,Unnamed: 0,geometry,object_id
0,02-026-00014,6.008675e+08,KULLU,2021_04,0.0,0.0,0.0,14.0,0.447610,0.336262,...,0.000447,3,3.0,5,4,0.722566,5,NaN,"POLYGON ((77.29864 32.13814, 77.29191 32.13627...",02-026-00014
1,02-023-00004,1.163670e+09,CHAMBA,2021_04,0.0,0.0,0.0,4.0,0.380152,0.238145,...,0.000395,2,3.0,5,4,0.644078,5,NaN,"POLYGON ((76.23815 33.02083, 76.22758 33.02138...",02-023-00004
2,02-024-02127,5.398900e+08,KANGRA,2021_04,599784.0,0.0,0.0,2127.0,0.201932,0.144395,...,0.000479,4,2.0,4,4,0.578561,4,NaN,"POLYGON ((75.93720 32.39733, 75.93302 32.39155...",02-024-02127
3,02-024-02110,2.928289e+08,KANGRA,2021_04,0.0,0.0,0.0,2110.0,0.320761,0.252522,...,0.000509,2,3.0,5,3,0.571543,4,NaN,"POLYGON ((76.73507 32.18612, 76.73014 32.18864...",02-024-02110
4,02-024-02116,2.778485e+08,KANGRA,2021_04,0.0,0.0,0.0,2116.0,0.201932,0.121925,...,0.000577,2,2.0,5,5,0.561715,4,NaN,"POLYGON ((75.96387 32.12435, 75.96410 32.12891...",02-024-02116
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5045,02-029,NaN,UNA,2024_04,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,4,3.0,5,3,NaN,4,9.0,None,NaN
5046,02-029,NaN,UNA,2024_05,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,4,3.0,5,3,NaN,4,9.0,None,NaN
5047,02-029,NaN,UNA,2024_06,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,4,3.0,5,3,NaN,4,9.0,None,NaN
5048,02-029,NaN,UNA,2024_07,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,4,3.0,5,3,NaN,4,9.0,None,NaN


In [9]:
merged_gdf['polygons'] = merged_gdf['geometry'].apply(geometry_to_text)
merged_gdf

,object-id,tehsil-area,district,timeperiod,total-tender-awarded-value,immediate-measures-tenders-awarded-value,others-tenders-awarded-value,objectid,max-rain,mean-rain,...,exposure,flood-hazard,government-response,vulnerability,topsis-score,risk-score,Unnamed: 0,geometry,object_id,polygons
0,02-026-00014,6.008675e+08,KULLU,2021_04,0.0,0.0,0.0,14.0,0.447610,0.336262,...,3,3.0,5,4,0.722566,5,NaN,"POLYGON ((77.29864 32.13814, 77.29191 32.13627...",02-026-00014,"[[77.29864218016306, 32.138139510562674], [77...."
1,02-023-00004,1.163670e+09,CHAMBA,2021_04,0.0,0.0,0.0,4.0,0.380152,0.238145,...,2,3.0,5,4,0.644078,5,NaN,"POLYGON ((76.23815 33.02083, 76.22758 33.02138...",02-023-00004,"[[76.23815327210251, 33.02082997060934], [76.2..."
2,02-024-02127,5.398900e+08,KANGRA,2021_04,599784.0,0.0,0.0,2127.0,0.201932,0.144395,...,4,2.0,4,4,0.578561,4,NaN,"POLYGON ((75.93720 32.39733, 75.93302 32.39155...",02-024-02127,"[[75.93719531891345, 32.39733490095737], [75.9..."
3,02-024-02110,2.928289e+08,KANGRA,2021_04,0.0,0.0,0.0,2110.0,0.320761,0.252522,...,2,3.0,5,3,0.571543,4,NaN,"POLYGON ((76.73507 32.18612, 76.73014 32.18864...",02-024-02110,"[[76.73507113671864, 32.18612437417027], [76.7..."
4,02-024-02116,2.778485e+08,KANGRA,2021_04,0.0,0.0,0.0,2116.0,0.201932,0.121925,...,2,2.0,5,5,0.561715,4,NaN,"POLYGON ((75.96387 32.12435, 75.96410 32.12891...",02-024-02116,"[[75.96386510035201, 32.12435128683737], [75.9..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5045,02-029,NaN,UNA,2024_04,NaN,NaN,NaN,NaN,NaN,NaN,...,4,3.0,5,3,NaN,4,9.0,None,NaN,None
5046,02-029,NaN,UNA,2024_05,NaN,NaN,NaN,NaN,NaN,NaN,...,4,3.0,5,3,NaN,4,9.0,None,NaN,None
5047,02-029,NaN,UNA,2024_06,NaN,NaN,NaN,NaN,NaN,NaN,...,4,3.0,5,3,NaN,4,9.0,None,NaN,None
5048,02-029,NaN,UNA,2024_07,NaN,NaN,NaN,NaN,NaN,NaN,...,4,3.0,5,3,NaN,4,9.0,None,NaN,None


In [10]:
merged_gdf = merged_gdf.dropna(subset =['object_id'])

In [12]:
merged_gdf.to_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\risk-score\HP\HP_risk.csv')

### Datetime ISO edits

In [11]:
from datetime import date, timedelta, datetime


#snapshot_ld_dist['datetime'] = pd.to_datetime(snapshot_ld_dist['timeperiod'], format='%Y_%m')
merged_gdf['timeperiod_iso'] = merged_gdf['timeperiod'].str.replace('_', '-') #+ '-01'

# Step 2: Convert the modified column to datetime format
#snapshot_ld_dist['timeperiod_iso'] = pd.to_datetime(snapshot_ld_dist['timeperiod_iso'], format='%Y-%m-%d')
merged_gdf['timeperiod_iso'] = pd.to_datetime(merged_gdf['timeperiod_iso'], format='%Y-%m')


merged_gdf['timeperiod_iso'] = merged_gdf['timeperiod_iso'].dt.strftime('%Y-%m')

C:\Users\saura\AppData\Local\Temp\ipykernel_79848\963400720.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_gdf['timeperiod_iso'] = merged_gdf['timeperiod'].str.replace('_', '-') #+ '-01'
C:\Users\saura\AppData\Local\Temp\ipykernel_79848\963400720.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_gdf['timeperiod_iso'] = pd.to_datetime(merged_gdf['timeperiod_iso'], format='%Y-%m')
C:\Users\saura\AppData\Local\Temp\ipykernel_79848\963400720.py:12: SettingWithCopyWarning: 
A value is trying

In [ ]:
merged_gdf.to_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\risk-score\risk_score_polygons.csv')